# Explainable Sentiment Analysis Using BERT
### A reproducible experimental study of predictive quality, attribution fidelity, and uncertainty

**Research notebook · October 2026 · End-to-end implementation**

| Dimension | Protocol |
|:--|:--|
| Task | Binary sentiment classification on the official IMDb movie-review benchmark |
| Encoder | `google-bert/bert-base-uncased` with a supervised classification head |
| Comparator | Word/bigram TF–IDF + logistic regression |
| Explanations | Layer Integrated Gradients (IG), Gradient × Activation, random ranking |
| Evaluation | Classification, bootstrap confidence intervals, temperature calibration, deletion/retention faithfulness, convergence, head-randomization check |
| Reproducibility | Fixed validation split, independent training seeds, frozen test set, exported artifacts |

> **Scope and scientific honesty:** This is a serious *research baseline and experimental framework*, **not** a new peer-reviewed method and not, by itself, an ICML-level contribution. A top-tier submission additionally needs a novel, falsifiable research claim; robust external/human-rationale evidence; ablations; sufficient compute; and independently repeated experiments. All numerical results in this notebook are **computed at execution time**—none are fabricated.

**Reading guide.** Run the notebook top-to-bottom with a CUDA GPU; the default is full protocol (`smoke_test=False`). Set `smoke_test=True` for a low-cost end-to-end functionality check only. Compute requirements and constraints are described below. All generated figures and CSV/JSON outputs are written into `outputs/`.

## 1. Research questions, preregistered hypotheses, and interpretation

**RQ1 (performance).** Does the fine-tuned BERT classifier outperform a strong n-gram baseline on **held-out IMDb reviews**? Compare paired per-review predictions with bootstrap intervals, rather than only reporting two accuracy numbers.

**RQ2 (explanation faithfulness).** For a *fixed BERT model* and predicted class, do words ranked highly by IG carry more predictive evidence than either Gradient × Activation or a length-matched random ranking? Use matched-size masking interventions, report **comprehensiveness** and **sufficiency**, and show entire budget-response curves rather than only a word heatmap.

**RQ3 (reliability).** How well calibrated are BERT probabilities, does validation-only temperature scaling help, and do errors vary across predeclared input-length/negation slices? A validation-tuned temperature is never fit to the test set.

**RQ4 (sanity).** Do explanations respond to a controlled model-weight intervention? We partially reinitialize the classification head and compute rank changes; *this is a diagnostic, not the full cascading parameter-randomization test*.

**Precommitted design:** three independent training seeds; 90/10 stratified development split from the official train portion; official test reserved for one final evaluation pass; fixed model/hyperparameters; test subset for explanation analysis sampled uniformly using a fixed RNG **before examining predictions**; review-level paired bootstraps; zero-shot sentence-level transfer on **SST-2 validation** using frozen IMDb-selected checkpoints (no adaptation). The main attribution analysis is conditional on a single explicitly chosen seed, so avoid claiming it generalizes across seeds without extending the evaluation.

**Endpoint directions:** ↑ accuracy, macro-F1, AUROC, comprehensiveness; ↓ Brier, negative log-likelihood, ECE, sufficiency, absolute IG completeness error. Comprehensiveness and sufficiency are perturbation-based *proxies* for faithfulness; [MASK] introduces distribution shift, so neither proves causal reasoning or plausibility.

**Core notation:** for trained classifier \(f\), target class \(c=\arg\max f(x)\), mask intervention \(M(x,R)\), and chosen rationale \(R\):

\[
\mathrm{Comp}(R)=p(c\mid x)-p(c\mid M(x,R)),\qquad
\mathrm{Suff}(R)=p(c\mid x)-p(c\mid M(x,\bar R)).
\]

We replace selected WordPiece IDs with `[MASK]` while **keeping attention masks and sequence positions constant**. This yields well-defined within-model comparisons but is not natural-language deletion. We rank **whitespace-independent word/punctuation groups** formed from character offsets, sum subtoken attributions in each group, and use identical cardinality budgets across methods. Comprehensiveness may be negative and sufficiency may be negative; do not clip them.

## 2. Related work and methodological grounding

- **BERT:** Devlin et al., *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding* (NAACL 2019), https://aclanthology.org/N19-1423/.
- **Integrated Gradients:** Sundararajan et al., *Axiomatic Attribution for Deep Networks* (ICML 2017), https://proceedings.mlr.press/v70/sundararajan17a.html. We attribute the predicted-class **logit**, not a probability, and track the completeness residual.
- **ERASER:** DeYoung et al., *ERASER: A Benchmark to Evaluate Rationalized NLP Models* (ACL 2020), https://aclanthology.org/2020.acl-main.408/. This motivates separate *plausibility* and *faithfulness* evaluation; the current IMDb data do **not** include gold rationales.
- **Sanity checks:** Adebayo et al., *Sanity Checks for Saliency Maps* (NeurIPS 2018), https://arxiv.org/abs/1810.03292. Visual saliency alone is not adequate evaluation.
- **IMDb provenance:** Maas et al., *Learning Word Vectors for Sentiment Analysis* (ACL 2011), https://aclanthology.org/P11-1015/; original dataset https://ai.stanford.edu/~amaas/data/sentiment/.
- **Implementation documentation:** https://captum.ai/api/layer.html and https://huggingface.co/docs/transformers/index.

This work uses **existing** attribution and rationale metrics. It does not claim that a heatmap exposes the model's internal deliberation; attributions estimate sensitivity or contribution under a specified computational and baseline choice.

## 3. Environment, resources, and configuration

**Recommended:** Python 3.10–3.12, CUDA GPU with ≥16 GB VRAM (or a larger GPU with bigger batches), about 15–30 GB disk for checkpoints/caches, internet for model/dataset download. Full three-seed fine-tuning and IG analysis can be substantial; CPU execution is technically possible but impractical. Reproducibility across different GPU kernels, drivers, and software versions is not bitwise guaranteed. Package version constraints are broad enough for compatible releases; record the resolved versions and lock them for a camera-ready experiment.

Run the installation cell once in a clean notebook environment, then restart the kernel if core modules were already imported. For restricted networks, pre-stage the Hugging Face datasets and model cache and disable download code.

In [ ]:
# Notebook dependencies. Uncomment and run if not already installed; restart the kernel afterward.
%pip install -q "torch>=2.3,<3" "transformers>=4.44,<6" "datasets>=3,<5" \
    "captum>=0.7,<1" "scikit-learn>=1.4,<2" "scipy>=1.11,<2" \
    "pandas>=2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" \
    "seaborn>=0.13,<1" "tqdm>=4.66,<5" "safetensors>=0.4,<1" \
    "ipywidgets>=8,<9"

In [ ]:
import os, sys, json, math, re, html, gc, time, random, hashlib, platform
from bisect import bisect_right
from dataclasses import dataclass, asdict
from pathlib import Path
from importlib.metadata import version

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp, softmax
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, log_loss, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)
from captum.attr import LayerIntegratedGradients, LayerGradientXActivation

plt.rcParams.update({"figure.dpi": 115, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
sns.set_theme(style="whitegrid", context="paper")

@dataclass(frozen=True)
class Config:
    model_name: str = "google-bert/bert-base-uncased"
    dataset_name: str = "stanfordnlp/imdb"
    split_seed: int = 2026
    train_seeds: tuple = (13, 42, 123)
    val_fraction: float = 0.10
    max_length: int = 256
    batch_size: int = 16
    eval_batch_size: int = 32
    grad_accum_steps: int = 2
    epochs: int = 3
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_fraction: float = 0.1
    patience: int = 2
    max_grad_norm: float = 1.0
    ig_steps: int = 32
    ig_internal_batch: int = 4
    explain_n: int = 64
    sanity_n: int = 8
    baseline_sensitivity_n: int = 12
    rationale_budgets: tuple = (0.10, 0.20, 0.30, 0.40)
    bootstrap_reps: int = 2000
    smoke_test: bool = False  # CHANGE ONLY FOR PIPELINE DEBUGGING; never publish smoke results.
    out_dir: str = "outputs"

CFG = Config()
if CFG.smoke_test:
    CFG = Config(smoke_test=True, train_seeds=(13,), epochs=1, max_length=128,
                 explain_n=8, sanity_n=3, baseline_sensitivity_n=3,
                 ig_steps=8, bootstrap_reps=100, batch_size=8, eval_batch_size=16)

OUT = Path(CFG.out_dir)
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
(OUT / "checkpoints").mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| config:", asdict(CFG))
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
assert CFG.max_length <= 512 and CFG.ig_internal_batch >= 1
(OUT / "config.json").write_text(json.dumps(asdict(CFG), indent=2))

VERSIONS = {p: version(p) for p in ["torch", "transformers", "datasets", "captum", "scikit-learn",
                                    "scipy", "numpy", "pandas", "matplotlib", "seaborn"]}
VERSIONS["python"] = platform.python_version()
VERSIONS["cuda_runtime"] = torch.version.cuda
(OUT / "environment.json").write_text(json.dumps(VERSIONS, indent=2))
display(pd.Series(VERSIONS, name="resolved version").to_frame())

## 4. Data acquisition and leakage controls

Use the **labeled** IMDb train and test splits. Normalize HTML line-break artifacts, remove exact duplicate training reviews, and deterministically exclude train reviews exactly matching official test text (without using test labels). Split only the cleaned *training pool* into train/validation with class stratification. Test reviews and labels are not used in training, checkpoint choice, temperature fitting, or explanation-method configuration. Dataset provenance and split text hashes are exported.

**Caveat:** exact-string deduplication does not identify paraphrases, reposts with formatting differences, near-duplicates, movie/author overlap, or pretraining contamination. IMDb is an established, relatively saturated benchmark; claims of broad deployment generality require other domains and newer datasets.

In [ ]:
def set_every_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def normalize_imdb(text: str) -> str:
    text = html.unescape(str(text))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def sha256_texts(texts):
    h = hashlib.sha256()
    for s in texts:
        h.update(s.encode("utf-8")); h.update(b"\x00")
    return h.hexdigest()

set_every_seed(CFG.split_seed)
raw = load_dataset(CFG.dataset_name)
train_pool = pd.DataFrame({"text": [normalize_imdb(s) for s in raw["train"]["text"]],
                           "label": list(map(int, raw["train"]["label"]))})
test_df = pd.DataFrame({"text": [normalize_imdb(s) for s in raw["test"]["text"]],
                        "label": list(map(int, raw["test"]["label"]))})
raw_train_n = len(train_pool)
train_pool = train_pool.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
dup_train_n = raw_train_n - len(train_pool)
exact_test_matches = train_pool["text"].isin(set(test_df["text"]))
train_test_overlap_n = int(exact_test_matches.sum())
train_pool = train_pool.loc[~exact_test_matches].reset_index(drop=True)

if CFG.smoke_test:
    # Balance-preserving subset of development pool and untouched-but-smaller test for debugging.
    take_dev, _ = train_test_split(np.arange(len(train_pool)), train_size=600,
                                   stratify=train_pool.label, random_state=CFG.split_seed)
    train_pool = train_pool.iloc[take_dev].reset_index(drop=True)
    take_test, _ = train_test_split(np.arange(len(test_df)), train_size=200,
                                   stratify=test_df.label, random_state=CFG.split_seed)
    test_df = test_df.iloc[take_test].reset_index(drop=True)

train_idx, val_idx = train_test_split(
    np.arange(len(train_pool)), test_size=CFG.val_fraction,
    stratify=train_pool["label"].to_numpy(), random_state=CFG.split_seed)
train_df = train_pool.iloc[train_idx].reset_index(drop=True)
val_df = train_pool.iloc[val_idx].reset_index(drop=True)
assert not set(train_df.text) & set(val_df.text)
assert not set(train_df.text) & set(test_df.text)
assert not set(val_df.text) & set(test_df.text)
assert set(train_df.label.unique()) == {0, 1}

counts = pd.DataFrame({"split": ["train", "validation", "test"],
                       "n": [len(train_df), len(val_df), len(test_df)],
                       "positive_rate": [x.label.mean() for x in [train_df, val_df, test_df]]})
provenance = {"dataset": CFG.dataset_name, "train_exact_deduplicated": dup_train_n,
              "removed_train_test_exact_overlaps": train_test_overlap_n,
              "train_text_sha256": sha256_texts(train_df.text),
              "validation_text_sha256": sha256_texts(val_df.text),
              "test_text_sha256": sha256_texts(test_df.text),
              "counts": counts.to_dict(orient="records")}
(OUT / "data_manifest.json").write_text(json.dumps(provenance, indent=2))
display(counts)
print("Removed", dup_train_n, "within-train duplicates and", train_test_overlap_n,
      "train/test exact overlaps, before development splitting.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.barplot(data=counts, x="split", y="n", ax=axes[0], color="steelblue")
axes[0].set(title="Examples by split", ylabel="review count", xlabel="")
train_lengths = train_df.text.str.split().str.len()
sns.histplot(train_lengths, bins=45, color="slateblue", ax=axes[1])
axes[1].set(title="Development training review lengths", xlabel="whitespace-delimited words")
fig.tight_layout()
fig.savefig(FIG / "01_data_profile.pdf", bbox_inches="tight")
plt.show()
print("Train word-length quantiles:", train_lengths.quantile([0.1, 0.5, 0.9, 0.99]).to_dict())

## 5. Shared tokenization, training protocol, and classical baseline

Use fast BERT WordPiece tokenization with **256-token head truncation** and dynamic padding. This is an explicit computational choice, not a guarantee of full-document coverage; evaluate truncation impacts in error analysis. The TF–IDF comparator sees the full text, so its input-length advantage must be disclosed. For a more controlled comparison, add a companion lexical baseline on the same truncated text (provided below).

Optimization: AdamW with no weight decay on bias and LayerNorm parameters; linear warmup/decay; gradient accumulation; clip global gradient norm; validation macro-F1 selects each seed's checkpoint. No hyperparameters are selected using test scores. We use ordinary FP32 training for portability and more predictable gradient attribution.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, use_fast=True)
assert tokenizer.is_fast, "Word-offset aggregation requires a fast tokenizer."
assert tokenizer.mask_token_id is not None and tokenizer.pad_token_id is not None

class ReviewDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.labels = df.label.astype(int).tolist()
        self.enc = tokenizer(df.text.tolist(), truncation=True, max_length=max_length,
                             padding=False, return_attention_mask=True)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {"input_ids": self.enc["input_ids"][idx],
                "attention_mask": self.enc["attention_mask"][idx],
                "labels": self.labels[idx]}

train_ds = ReviewDataset(train_df, tokenizer, CFG.max_length)
val_ds = ReviewDataset(val_df, tokenizer, CFG.max_length)
test_ds = ReviewDataset(test_df, tokenizer, CFG.max_length)
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")

def make_loader(ds, batch_size, shuffle=False, seed=0):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collator, generator=generator, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def make_model():
    return AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name, num_labels=2,
        id2label={0: "negative", 1: "positive"},
        label2id={"negative": 0, "positive": 1}).to(DEVICE)

def forward_logits(model, ds, batch_size=None):
    model.eval(); logits = []; labels = []
    loader = make_loader(ds, batch_size or CFG.eval_batch_size)
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Inference", leave=False):
            labels.append(batch["labels"].cpu().numpy())
            x = to_device(batch)
            logits.append(model(**x).logits.float().cpu().numpy())
    return np.concatenate(logits), np.concatenate(labels)

# Confirm input type, model vocabulary alignment, and class mapping.
check_batch = next(iter(make_loader(train_ds, 3)))
assert check_batch["input_ids"].max().item() < tokenizer.vocab_size
assert set(check_batch) == {"input_ids", "attention_mask", "labels"}
print("Batch shape:", tuple(check_batch["input_ids"].shape),
      "| special IDs:", tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.mask_token_id)

In [ ]:
# Full-review and BERT-truncated-input lexical controls (same train/val/test partitions).
def truncated_texts(df):
    ids = tokenizer(df.text.tolist(), truncation=True, max_length=CFG.max_length,
                    padding=False)["input_ids"]
    return tokenizer.batch_decode(ids, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)

lexical_data = {
    "full": (train_df.text.tolist(), val_df.text.tolist(), test_df.text.tolist()),
    "bert_truncated": (truncated_texts(train_df), truncated_texts(val_df), truncated_texts(test_df))
}

def fit_lexical_control(train_text, val_text, test_text):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000,
                                 sublinear_tf=True, strip_accents="unicode")
    xtr = vectorizer.fit_transform(train_text)
    clf = LogisticRegression(C=1.0, max_iter=1500, random_state=CFG.split_seed)
    clf.fit(xtr, train_df.label.to_numpy())
    return {"vectorizer": vectorizer, "clf": clf,
            "val_proba": clf.predict_proba(vectorizer.transform(val_text)),
            "test_proba": clf.predict_proba(vectorizer.transform(test_text))}

lexical = {name: fit_lexical_control(*text_sets)
           for name, text_sets in lexical_data.items()}
print("Lexical control features:",
      {name: len(obj["vectorizer"].vocabulary_) for name, obj in lexical.items()})

## 6. Fine-tune BERT with independent random seeds

**Selection rule:** highest validation macro-F1; if equal, retain the earlier epoch. Epoch-level validation loss is recorded but not independently optimized. All test predictions are deferred until **after** every seed's checkpoint is fixed. The optimizer schedule is recomputed for every seed. A research extension should include repeated data splits or cross-domain validation; here the development split is held fixed so training variance is isolated.

In [ ]:
def build_optimizer(model):
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": CFG.weight_decay},
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0}
    ]
    return torch.optim.AdamW(groups, lr=CFG.lr)

def train_one_seed(seed):
    set_every_seed(seed)
    model = make_model()
    train_loader = make_loader(train_ds, CFG.batch_size, shuffle=True, seed=seed)
    optimizer = build_optimizer(model)
    total_updates = math.ceil(len(train_loader) / CFG.grad_accum_steps) * CFG.epochs
    warmup = round(total_updates * CFG.warmup_fraction)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_updates)
    best_f1, stale_epochs, history = -float("inf"), 0, []
    ckpt = OUT / "checkpoints" / f"bert_seed_{seed}"
    ckpt.mkdir(parents=True, exist_ok=True)
    start = time.time()
    for epoch in range(1, CFG.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        it = tqdm(enumerate(train_loader), total=len(train_loader),
                  desc=f"seed={seed} epoch={epoch}")
        for step, batch in it:
            x = to_device(batch)
            output = model(**x)
            # Correctly divide the final short accumulation window.
            window_start = (step // CFG.grad_accum_steps) * CFG.grad_accum_steps
            actual_window = min(CFG.grad_accum_steps, len(train_loader) - window_start)
            (output.loss / actual_window).backward()
            running_loss += output.loss.item()
            if (step + 1) % CFG.grad_accum_steps == 0 or step + 1 == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
            if (step + 1) % 100 == 0:
                it.set_postfix(train_loss=f"{running_loss / (step + 1):.3f}")
        val_logits, y_val = forward_logits(model, val_ds)
        val_pred = val_logits.argmax(axis=1)
        metrics = {"seed": seed, "epoch": epoch,
                   "train_loss": running_loss / max(1, len(train_loader)),
                   "val_macro_f1": float(f1_score(y_val, val_pred, average="macro")),
                   "val_accuracy": float(accuracy_score(y_val, val_pred)),
                   "val_nll": float(log_loss(y_val, softmax(val_logits, axis=1), labels=[0, 1]))}
        history.append(metrics)
        print(metrics)
        if metrics["val_macro_f1"] > best_f1:
            best_f1 = metrics["val_macro_f1"]
            stale_epochs = 0
            model.save_pretrained(ckpt, safe_serialization=True)
            tokenizer.save_pretrained(ckpt)
            (ckpt / "selection.json").write_text(json.dumps(metrics, indent=2))
        else:
            stale_epochs += 1
        if stale_epochs >= CFG.patience:
            print(f"Early stopping for seed {seed} after epoch {epoch}")
            break
    del model, optimizer, scheduler
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print(f"Seed {seed} completed in {(time.time() - start) / 60:.1f} minutes")
    return history

all_history = []
for seed in CFG.train_seeds:
    all_history.extend(train_one_seed(seed))
history_df = pd.DataFrame(all_history)
history_df.to_csv(OUT / "training_history.csv", index=False)
display(history_df)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for seed, g in history_df.groupby("seed"):
    axes[0].plot(g.epoch, g.train_loss, marker="o", label=f"seed {seed}")
    axes[1].plot(g.epoch, g.val_macro_f1, marker="o", label=f"seed {seed}")
axes[0].set(xlabel="epoch", ylabel="loss", title="Optimization")
axes[1].set(xlabel="epoch", ylabel="validation macro-F1", title="Model selection")
axes[1].legend()
fig.tight_layout(); fig.savefig(FIG / "02_training.pdf", bbox_inches="tight"); plt.show()

## 7. Probabilistic evaluation and validation-only temperature scaling

For temperature \(T>0\), calibrated probabilities are \(\operatorname{softmax}(z/T)\). Optimize \(\log T\) on **validation NLL** (not test). This transforms confidence without changing the predicted class. Report discrimination and calibration separately: Brier score and NLL reflect both, while binned expected calibration error (ECE) is an imperfect discretized diagnostic.

All neural test sets are evaluated **after** checkpoint and temperature selection. Uncertainty is quantified later with review-level paired resampling and between-seed variability.

In [ ]:
def fit_temperature(val_logits, y_val):
    val_logits = np.asarray(val_logits, dtype=np.float64)
    y_val = np.asarray(y_val, dtype=int)
    def objective(log_temp):
        scaled = val_logits / np.exp(float(log_temp))
        return float(np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(y_val)), y_val]))
    fit = minimize_scalar(objective, method="bounded", bounds=(-4, 4),
                          options={"xatol": 1e-5})
    assert fit.success
    return float(np.exp(fit.x))

def top_label_ece(y, proba, n_bins=15):
    y = np.asarray(y); p = np.asarray(proba)
    conf = p.max(axis=1); correct = (p.argmax(axis=1) == y)
    boundaries = np.linspace(0, 1, n_bins + 1)
    bins = np.minimum(np.searchsorted(boundaries[1:], conf, side="right"), n_bins - 1)
    parts = []
    for b in range(n_bins):
        ix = bins == b
        if ix.any():
            parts.append({"n": int(ix.sum()), "confidence": float(conf[ix].mean()),
                          "accuracy": float(correct[ix].mean()), "bin": b})
    ece = sum(row["n"] * abs(row["confidence"] - row["accuracy"])
              for row in parts) / len(y)
    return float(ece), pd.DataFrame(parts)

def classification_metrics(y, proba):
    y = np.asarray(y, dtype=int)
    p = np.asarray(proba, dtype=float)
    pred = p.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "auroc": float(roc_auc_score(y, p[:, 1])) if len(np.unique(y)) == 2 else float("nan"),
        "brier": float(brier_score_loss(y, p[:, 1])),
        "nll": float(log_loss(y, p, labels=[0, 1])),
        "ece_15": float(top_label_ece(y, p)[0])
    }

results = []
seed_outputs = {}
y_test = test_df.label.to_numpy(dtype=int)
y_val = val_df.label.to_numpy(dtype=int)
for seed in CFG.train_seeds:
    # Load the selected checkpoint; no more gradient-based model updates.
    model = AutoModelForSequenceClassification.from_pretrained(
        OUT / "checkpoints" / f"bert_seed_{seed}").to(DEVICE)
    val_logits, y_val_check = forward_logits(model, val_ds)
    test_logits, y_test_check = forward_logits(model, test_ds)
    assert np.array_equal(y_val_check, y_val) and np.array_equal(y_test_check, y_test)
    temperature = fit_temperature(val_logits, y_val)
    calibrated = softmax(test_logits / temperature, axis=1)
    uncalibrated = softmax(test_logits, axis=1)
    for stage, p in [("raw", uncalibrated), ("temperature_scaled", calibrated)]:
        results.append({"system": "BERT", "seed": seed, "calibration": stage,
                        **classification_metrics(y_test, p)})
    seed_outputs[seed] = {"val_logits": val_logits, "test_logits": test_logits,
                          "test_proba": calibrated, "temperature": temperature}
    del model
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

for name, obj in lexical.items():
    results.append({"system": f"TFIDF_{name}", "seed": "fixed", "calibration": "raw",
                    **classification_metrics(y_test, obj["test_proba"])})
results_df = pd.DataFrame(results)
results_df.to_csv(OUT / "classification_results.csv", index=False)
display(results_df.round(4))
print("Temperatures:", {s: round(v["temperature"], 4) for s,v in seed_outputs.items()})

In [ ]:
# Reliability diagram for predeclared explanation seed (raw vs validation-calibrated).
EXPLAIN_SEED = CFG.train_seeds[0]
logits_main = seed_outputs[EXPLAIN_SEED]["test_logits"]
raw_p = softmax(logits_main, axis=1)
cal_p = seed_outputs[EXPLAIN_SEED]["test_proba"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, proba in [("raw", raw_p), ("temperature-scaled", cal_p)]:
    ece, bins = top_label_ece(y_test, proba)
    axes[0].plot(bins.confidence, bins.accuracy, marker="o", label=f"{name} (ECE={ece:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="mean top-class confidence",
            ylabel="empirical accuracy", title="Reliability")
axes[0].legend(fontsize=8)
ConfusionMatrixDisplay(confusion_matrix(y_test, cal_p.argmax(axis=1)),
                       display_labels=["negative", "positive"]).plot(
                           ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set(title=f"Held-out confusion matrix | seed {EXPLAIN_SEED}")
fig.tight_layout(); fig.savefig(FIG / "03_reliability_confusion.pdf", bbox_inches="tight")
plt.show()
print(classification_report(y_test, cal_p.argmax(axis=1), target_names=["negative", "positive"]))

## 8. Statistical inference (paired review-level bootstrap)

Sample reviews **with replacement** from the same test set and recompute each metric within a replicate. The accuracy-difference interval is *paired* because both systems are evaluated on the same sampled reviews. This represents sampling uncertainty conditional on a fixed trained model, **not** uncertainty due to seeds, datasets, hyperparameter tuning, or deployment drift. Report across-seed mean ± SD separately; do not regard three seeds as a sufficiently powered hypothesis test. Descriptive intervals are not proof of significance unless a pre-specified testing protocol supports that inference.

In [ ]:
def bootstrap_metric(y, p, metric_fn, reps=2000, seed=101):
    rng = np.random.default_rng(seed)
    n = len(y)
    values = np.empty(reps)
    for i in range(reps):
        take = rng.integers(0, n, size=n)
        values[i] = metric_fn(y[take], p[take])
    return np.quantile(values, [0.025, 0.5, 0.975])

def paired_bootstrap_acc_delta(y, p_a, p_b, reps=2000, seed=101):
    a = (np.asarray(p_a).argmax(1) == y).astype(float)
    b = (np.asarray(p_b).argmax(1) == y).astype(float)
    d = a - b
    rng = np.random.default_rng(seed)
    samples = np.empty(reps)
    for i in range(reps):
        samples[i] = d[rng.integers(0, len(d), size=len(d))].mean()
    return np.quantile(samples, [0.025, 0.5, 0.975])

baseline_full = lexical["full"]["test_proba"]
ci_rows = []
for seed in CFG.train_seeds:
    probs = seed_outputs[seed]["test_proba"]
    ci = bootstrap_metric(y_test, probs,
                          lambda y,p: accuracy_score(y, p.argmax(1)),
                          CFG.bootstrap_reps, seed=2026)
    delta_ci = paired_bootstrap_acc_delta(y_test, probs, baseline_full,
                                          CFG.bootstrap_reps, seed=2026)
    ci_rows.append({"seed": seed, "accuracy_2.5%": ci[0], "accuracy_median": ci[1],
                    "accuracy_97.5%": ci[2], "delta_vs_full_tfidf_2.5%": delta_ci[0],
                    "delta_median": delta_ci[1], "delta_97.5%": delta_ci[2]})
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT / "bootstrap_summary.csv", index=False)
display(ci_df.round(4))
bert_cal = results_df.query("system == 'BERT' and calibration == 'temperature_scaled'")
if len(bert_cal) > 1:
    display(bert_cal[["accuracy", "macro_f1", "brier", "ece_15"]]
            .agg(["mean", "std"]).rename(index={"std": "sample SD across training seeds"}).round(4))

## 9. Explanation algorithms and a word-level alignment contract

**Input representation.** Let \(z_i\) denote an embedding vector at a BERT input position. With reference (baseline) embeddings \(z'_i\), Layer IG computes

\[
\mathrm{IG}_{i}=\sum_d (z_{id}-z'_{id})
\int_{0}^{1}\frac{\partial f_c(z'+\alpha(z-z'))}{\partial z_{id}}\,d\alpha.
\]

The implementation approximates this integral with Gauss–Legendre quadrature, holding attention masks fixed. The **baseline is `[PAD]` IDs in content positions with the original attention mask left unchanged**; that is a synthetic reference input (PAD embeddings are attended to). Special tokens are unchanged. The chosen embedding layer includes token, position, and token-type vectors, while only the token IDs vary between endpoints. We also test a `[MASK]` reference on a subset as a baseline-sensitivity diagnostic.

**Comparators.** Gradient × Activation is a first-order local gradient-based method, *not* the same attribution axioms as IG. A random word ranking is a cardinality-matched negative control. All methods rank **signed positive evidence for the predicted class** (not magnitude), which matters for faithful removal; using absolute saliency would answer a different question.

**Alignment.** Fast-tokenizer character offsets are mapped to regex-defined word/punctuation spans, and WordPiece scores belonging to a span are *summed*. No lowercasing of original displayed text is required. Punctuation is a separate group and is eligible for perturbation. Only tokens inside BERT's truncated window can receive attribution. The code asserts that every non-special token maps to exactly one group, preventing silent token loss.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    OUT / "checkpoints" / f"bert_seed_{EXPLAIN_SEED}").to(DEVICE)
model.eval()
# Explicit dependency on a BERT-family model; not portable to arbitrary HF architectures.
assert hasattr(model, "bert") and hasattr(model.bert, "embeddings")

def bert_forward_logits(input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

ig = LayerIntegratedGradients(bert_forward_logits, model.bert.embeddings)
gxa = LayerGradientXActivation(bert_forward_logits, model.bert.embeddings)

def encode_for_explanation(text):
    batch = tokenizer(text, truncation=True, max_length=CFG.max_length,
                      return_offsets_mapping=True, return_special_tokens_mask=True,
                      return_tensors="pt")
    offsets = batch.pop("offset_mapping")[0].tolist()
    special_mask = batch.pop("special_tokens_mask")[0].tolist()
    return ({k: v.to(DEVICE) for k, v in batch.items()}, offsets, special_mask)

def reference_ids(ids, special_mask, reference="pad"):
    reference_token = tokenizer.pad_token_id if reference == "pad" else tokenizer.mask_token_id
    ref = ids.clone()
    for i, special in enumerate(special_mask):
        if not special: ref[0, i] = reference_token
    return ref

WORD_RE = re.compile(r"\w+(?:['’]\w+)*|[^\w\s]", flags=re.UNICODE)

def aggregate_word_scores(text, offsets, special_mask, token_scores):
    spans = list(WORD_RE.finditer(text))
    starts = [m.start() for m in spans]
    groups = [{"word": m.group(), "start": m.start(), "end": m.end(),
               "token_positions": [], "score": 0.0} for m in spans]
    for pos, ((start, end), special) in enumerate(zip(offsets, special_mask)):
        if special or end <= start: continue
        word_idx = bisect_right(starts, start) - 1
        if word_idx < 0 or not (spans[word_idx].start() <= start < spans[word_idx].end()):
            raise ValueError(f"Unaligned tokenizer span {(start, end)} at pos {pos}")
        groups[word_idx]["token_positions"].append(pos)
        groups[word_idx]["score"] += float(token_scores[pos])
    return [g for g in groups if g["token_positions"]]

def explain_text(text, reference="pad", ig_steps=None):
    model.eval()
    x, offsets, special_mask = encode_for_explanation(text)
    ids, mask = x["input_ids"], x["attention_mask"]
    ref = reference_ids(ids, special_mask, reference)
    with torch.no_grad():
        pred = int(bert_forward_logits(ids, mask).argmax(dim=-1).item())
    attr, convergence_delta = ig.attribute(
        inputs=ids, baselines=ref, additional_forward_args=(mask,),
        target=pred, n_steps=ig_steps or CFG.ig_steps,
        internal_batch_size=CFG.ig_internal_batch, return_convergence_delta=True)
    # Signed class-logit contributions, summing embedding dimensions.
    ig_scores = attr.sum(dim=-1).detach().cpu().numpy()[0]
    local_attr = gxa.attribute(inputs=ids, additional_forward_args=(mask,), target=pred)
    gx_scores = local_attr.sum(dim=-1).detach().cpu().numpy()[0]
    group_ig = aggregate_word_scores(text, offsets, special_mask, ig_scores)
    group_gx = aggregate_word_scores(text, offsets, special_mask, gx_scores)
    assert [g["token_positions"] for g in group_ig] == [g["token_positions"] for g in group_gx]
    assert sum(len(g["token_positions"]) for g in group_ig) == sum(
        (not s and b > a) for (a,b),s in zip(offsets,special_mask))
    return {"prediction": pred, "ig_groups": group_ig,
            "gxa_groups": group_gx, "ids": ids.detach().cpu(),
            "mask": mask.detach().cpu(), "special_mask": special_mask,
            "ig_abs_delta": abs(float(convergence_delta.detach().cpu().item()))}

In [ ]:
# Fast dependency-free checks of the alignment and perturbation assumptions.
assert normalize_imdb("Good<br /> movie") == "Good movie"
toy_text = "Unbelievably good!"
toy_groups = aggregate_word_scores(toy_text, [(0,0),(0,2),(2,12),(13,17),(17,18),(0,0)],
                                    [1,0,0,0,0,1], np.array([0,0.2,0.3,0.4,0.1,0]))
assert [g["word"] for g in toy_groups] == ["Unbelievably", "good", "!"]
assert np.isclose(toy_groups[0]["score"], 0.5)
assert len(toy_groups[0]["token_positions"]) == 2
print("Alignment assertions passed.")

## 10. Qualitative review explanations

We sample cases uniformly using a fixed RNG—**not by cherry-picking correct, confident, or visually attractive predictions**. In the heatmaps, positive values are evidence for the *predicted* class and negative values are opposing evidence. Explanations are local and conditional on the trained checkpoint and baseline. HTML visualization retains original whitespace and highlights the **first 100 groups** for legibility (all tokens still count in numerical evaluation).

In [ ]:
from IPython.display import display, HTML

EXPLAIN_INDEX = np.random.default_rng(2026).choice(
    len(test_df), size=min(CFG.explain_n, len(test_df)), replace=False)
assert len(EXPLAIN_INDEX) == len(np.unique(EXPLAIN_INDEX))


def render_word_highlights(text, groups, limit=100):
    groups = groups[:limit]
    max_abs = max([abs(g["score"]) for g in groups] + [1e-10])
    fragments, last = [], 0
    for g in groups:
        fragments.append(html.escape(text[last:g["start"]]))
        alpha = min(0.75, abs(g["score"]) / max_abs * 0.75)
        rgb = "56, 142, 60" if g["score"] >= 0 else "229, 57, 53"
        fragments.append(f'<span title="IG={g["score"]:+.4f}" '
                         f'style="background:rgba({rgb},{alpha:.3f}); padding:1px;">'
                         f'{html.escape(text[g["start"]:g["end"]])}</span>')
        last = g["end"]
    if groups: fragments.append(html.escape(text[last:groups[-1]["end"]]))
    return HTML('<div style="line-height:1.8; max-width:950px">' + ''.join(fragments) + '</div>')

example_cache = {}
for idx in EXPLAIN_INDEX[:min(3, len(EXPLAIN_INDEX))]:
    idx = int(idx)
    text = test_df.iloc[idx].text
    explanation = explain_text(text)
    example_cache[idx] = explanation
    print(f"test index={idx} | truth={['negative','positive'][test_df.iloc[idx].label]} "
          f"| prediction={['negative','positive'][explanation['prediction']]} "
          f"| |completeness residual|={explanation['ig_abs_delta']:.5f}")
    display(render_word_highlights(text, explanation["ig_groups"]))

## 11. Quantitative explanation faithfulness: matched-budget interventions

For every uniformly sampled review, compute three rankings: IG, Gradient × Activation, and one seeded random permutation. For each rationale budget \(b\in\{10\%,20\%,30\%,40\%\}\), round up to at least one word group; every method then masks/keeps exactly the same **number of groups for that review**. This avoids the common confound where one method simply selects more tokens.

- **Comprehensiveness (↑):** reduce predicted-class probability by masking the chosen groups.
- **Sufficiency (↓):** preserve predicted-class probability by keeping only those groups and masking all other non-special groups.
- **AOPC:** average each curve over the predeclared budgets. Do not compare raw AOPC on substantially different dataset collections.
- **Uncertainty:** paired bootstrap over **review-level** AOPC differences (IG minus controls) instead of treating multiple budgets per review as independent samples.

Interpret the random control as a *null ranking under the same intervention*, not a randomized classifier. MASKing changes the input distribution and can produce nonmonotone curves; negative or surprising scores must remain visible.

In [ ]:
def rank_groups(groups):
    return sorted(range(len(groups)), key=lambda i: (-groups[i]["score"], i))

def intervention_ids(original_ids, groups, chosen, keep_only=False):
    ids = original_ids.clone()
    chosen = set(chosen)
    for j, group in enumerate(groups):
        should_mask = (j not in chosen) if keep_only else (j in chosen)
        if should_mask:
            ids[0, group["token_positions"]] = tokenizer.mask_token_id
    return ids

@torch.inference_mode()
def predict_target_for_variants(variants, attention_mask, target, batch_size=16):
    model.eval(); values = []
    for left in range(0, len(variants), batch_size):
        batch_ids = torch.cat(variants[left:left + batch_size], dim=0).to(DEVICE)
        batch_mask = attention_mask.expand(len(batch_ids), -1).to(DEVICE)
        probabilities = model(input_ids=batch_ids, attention_mask=batch_mask).logits.softmax(dim=-1)
        values.extend(probabilities[:, target].detach().cpu().tolist())
    return np.asarray(values)

# Intervention contract: no special token or unselected token should change.
_test_ids = torch.tensor([[101, 2054, 2003, 102]])
_test_groups = [{"token_positions": [1]}, {"token_positions": [2]}]
assert intervention_ids(_test_ids, _test_groups, [0])[0,1].item() == tokenizer.mask_token_id
assert intervention_ids(_test_ids, _test_groups, [0])[0,0].item() == 101
assert intervention_ids(_test_ids, _test_groups, [0], keep_only=True)[0,2].item() == tokenizer.mask_token_id

records, deltas = [], []
for idx in tqdm(EXPLAIN_INDEX, desc="Attribution + faithfulness"):
    idx = int(idx)
    exp = example_cache.get(idx) or explain_text(test_df.iloc[idx].text)
    groups = exp["ig_groups"]
    if len(groups) == 0: continue
    rankings = {
        "IntegratedGradients": rank_groups(groups),
        "GradientXActivation": rank_groups(exp["gxa_groups"]),
        "Random": np.random.default_rng(100000 + idx).permutation(len(groups)).tolist()
    }
    ids, mask, target = exp["ids"], exp["mask"], exp["prediction"]
    variants = [ids]
    specs = []
    for method, ordering in rankings.items():
        for budget in CFG.rationale_budgets:
            k = min(len(groups), max(1, math.ceil(budget * len(groups))))
            selected = ordering[:k]
            variants.append(intervention_ids(ids, groups, selected, keep_only=False))
            variants.append(intervention_ids(ids, groups, selected, keep_only=True))
            specs.append((method, budget, k))
    probs = predict_target_for_variants(variants, mask, target)
    p_full = probs[0]
    for j, (method, budget, k) in enumerate(specs):
        p_drop, p_keep = probs[1 + 2*j], probs[2 + 2*j]
        records.append({"test_index": idx, "method": method, "budget": budget,
                        "group_count": len(groups), "selected_groups": k,
                        "p_original": p_full, "comprehensiveness": p_full - p_drop,
                        "sufficiency": p_full - p_keep,
                        "correct": bool(target == test_df.iloc[idx].label)})
    deltas.append({"test_index": idx, "ig_abs_delta": exp["ig_abs_delta"]})

faith_df = pd.DataFrame(records)
delta_df = pd.DataFrame(deltas)
faith_df.to_csv(OUT / "faithfulness_review_budget.csv", index=False)
delta_df.to_csv(OUT / "ig_completeness.csv", index=False)
assert faith_df.groupby(["test_index", "budget"]).selected_groups.nunique().max() == 1
assert np.isfinite(faith_df[["comprehensiveness", "sufficiency"]].to_numpy()).all()
print("Mean |IG completeness residual|:", delta_df.ig_abs_delta.mean())
display(faith_df.groupby(["method", "budget"])[["comprehensiveness", "sufficiency"]]
        .mean().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.8), sharex=True)
for method, g in faith_df.groupby("method"):
    curve = g.groupby("budget")[["comprehensiveness", "sufficiency"]].mean().reset_index()
    for a, key, title in zip(axes, ["comprehensiveness", "sufficiency"],
                             ["Comprehensiveness ↑", "Sufficiency ↓"]):
        a.plot(curve.budget, curve[key], marker="o", label=method)
        a.set(title=title, xlabel="fraction of word groups", ylabel="mean probability difference")
axes[1].legend(loc="best", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "04_faithfulness_curves.pdf", bbox_inches="tight"); plt.show()

# Review-level mean across budgets. Review is the independent sampling unit.
aopc = faith_df.groupby(["test_index", "method"], as_index=False)[
    ["comprehensiveness", "sufficiency"]].mean()
aopc.to_csv(OUT / "faithfulness_review_aopc.csv", index=False)

def paired_mean_ci(x, y, reps=2000, seed=2026):
    diff = np.asarray(x) - np.asarray(y)
    assert len(diff) > 0
    rng = np.random.default_rng(seed)
    boot = np.array([diff[rng.integers(0, len(diff), size=len(diff))].mean()
                     for _ in range(reps)])
    return float(diff.mean()), *np.quantile(boot, [0.025, 0.975]).tolist()

summary_rows = []
for metric in ["comprehensiveness", "sufficiency"]:
    wide = aopc.pivot(index="test_index", columns="method", values=metric).dropna()
    for control in ["Random", "GradientXActivation"]:
        effect, lower, upper = paired_mean_ci(wide["IntegratedGradients"], wide[control],
                                              CFG.bootstrap_reps)
        summary_rows.append({"metric": metric, "comparison": f"IG - {control}",
                             "paired_difference": effect, "CI_lower": lower,
                             "CI_upper": upper, "n_reviews": len(wide)})
faith_comparison = pd.DataFrame(summary_rows)
faith_comparison.to_csv(OUT / "faithfulness_paired_intervals.csv", index=False)
display(faith_comparison.round(4))

## 12. Two explanation diagnostics: reference choice and weight randomization

**Reference sensitivity:** rerun IG with `[MASK]` rather than `[PAD]` reference IDs on a **fixed subset** and compare signed word-score rank correlations. Attribution is reference-dependent; disagreement should be reported, not hidden.

**Head randomization:** temporarily replace the trained classification head's weights and recompute IG for the *same original target class*. Compare original and intervened signed word rankings. Restoration is guaranteed with `finally`. This is a limited **single-layer** intervention; it does **not** reproduce the full Adebayo et al. cascading randomization sequence or label-randomization training test. Pearson/Spearman rank similarity near 1 after randomization would be a warning that a purportedly model-dependent explanation is insensitive to a learned component.

In [ ]:
sensitivity_rows = []
for idx in tqdm(EXPLAIN_INDEX[:min(CFG.baseline_sensitivity_n, len(EXPLAIN_INDEX))],
                desc="IG reference sensitivity"):
    idx = int(idx)
    orig = example_cache.get(idx) or explain_text(test_df.iloc[idx].text)
    alternate = explain_text(test_df.iloc[idx].text, reference="mask")
    original_scores = np.array([g["score"] for g in orig["ig_groups"]])
    other_scores = np.array([g["score"] for g in alternate["ig_groups"]])
    assert original_scores.shape == other_scores.shape
    rho = spearmanr(original_scores, other_scores).statistic if len(original_scores) > 2 else np.nan
    sensitivity_rows.append({"test_index": idx, "signed_spearman": rho,
                             "pad_abs_delta": orig["ig_abs_delta"],
                             "mask_abs_delta": alternate["ig_abs_delta"]})
sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(OUT / "ig_reference_sensitivity.csv", index=False)
display(sensitivity_df.round(3))

# Classifier-head-only perturbation. Captum references this same model object.
head = model.classifier
assert isinstance(head, nn.Linear), "This diagnostic specifically assumes BERT's Linear classifier."
orig_weight = head.weight.detach().clone()
orig_bias = head.bias.detach().clone() if head.bias is not None else None
randomization_rows = []
try:
    set_every_seed(987654)
    with torch.no_grad():
        nn.init.normal_(head.weight, mean=0, std=0.02)
        if head.bias is not None: nn.init.zeros_(head.bias)
    for idx in tqdm(EXPLAIN_INDEX[:min(CFG.sanity_n, len(EXPLAIN_INDEX))],
                    desc="Randomized-head IG"):
        idx = int(idx)
        original = example_cache.get(idx) or explain_text(test_df.iloc[idx].text)
        # Force the ORIGINAL model's predicted class, not the randomized head's new class.
        text = test_df.iloc[idx].text
        x, offsets, special = encode_for_explanation(text)
        base = reference_ids(x["input_ids"], special, "pad")
        attributions = ig.attribute(inputs=x["input_ids"], baselines=base,
                                    additional_forward_args=(x["attention_mask"],),
                                    target=original["prediction"], n_steps=CFG.ig_steps,
                                    internal_batch_size=CFG.ig_internal_batch)
        randomized = aggregate_word_scores(text, offsets, special,
                            attributions.sum(-1).detach().cpu().numpy()[0])
        a = np.array([g["score"] for g in original["ig_groups"]])
        b = np.array([g["score"] for g in randomized])
        assert a.shape == b.shape
        rho = spearmanr(a, b).statistic if len(a) > 2 else np.nan
        randomization_rows.append({"test_index": idx, "signed_spearman": rho,
                                   "original_class": original["prediction"]})
finally:
    with torch.no_grad():
        head.weight.copy_(orig_weight)
        if orig_bias is not None: head.bias.copy_(orig_bias)
    model.eval()

randomization_df = pd.DataFrame(randomization_rows)
randomization_df.to_csv(OUT / "head_randomization.csv", index=False)
display(randomization_df.round(3))

## 13. Error analysis and predeclared diagnostic slices

Slices use only **observed text characteristics**: whitespace word count, presence of explicit negation, and token-budget truncation. These are **descriptive**, not fairness guarantees or independent causal explanations. We refrain from estimating demographic fairness because IMDb does not have validated demographic annotations and inferring protected traits from review text would be inappropriate. Small slices get count warnings and should not be overinterpreted.

A further concern is the *long-document truncation problem*: a review may contain a sentiment reversal after token 256, and BERT cannot use text it never sees.

In [ ]:
full_word_counts = test_df.text.str.split().str.len().to_numpy()
# Token lengths without truncation are for analysis only (not BERT inference).
full_token_lengths = np.array([len(ids) for ids in tokenizer(
    test_df.text.tolist(), truncation=False, padding=False)["input_ids"]])
negation = test_df.text.str.contains(r"\b(?:not|never|no|neither|nor|n't)\b",
                                     flags=re.IGNORECASE, regex=True).to_numpy()
correct = (cal_p.argmax(1) == y_test)

slices = {
    "short <= 100 words": full_word_counts <= 100,
    "medium 101–300 words": (full_word_counts > 100) & (full_word_counts <= 300),
    "long > 300 words": full_word_counts > 300,
    "contains explicit negation": negation,
    "no explicit negation": ~negation,
    "truncated by BERT": full_token_lengths > CFG.max_length,
    "not truncated": full_token_lengths <= CFG.max_length,
}
rows = []
for name, subset in slices.items():
    n = int(subset.sum())
    if n == 0: continue
    rows.append({"slice": name, "n": n, "accuracy": float(correct[subset].mean()),
                 "positive_rate": float(y_test[subset].mean()),
                 "mean_confidence": float(cal_p[subset].max(axis=1).mean()),
                 "reliable_sample": n >= 100})
slice_df = pd.DataFrame(rows)
slice_df.to_csv(OUT / "error_slices.csv", index=False)
display(slice_df.round(3))
print("BERT truncated proportion of reviews:", round((full_token_lengths > CFG.max_length).mean(), 3))

# Fixed, reproducible sample of errors for responsible manual inspection.
wrong_idx = np.flatnonzero(~correct)
error_rng = np.random.default_rng(999)
sampled_errors = error_rng.choice(wrong_idx, min(10, len(wrong_idx)), replace=False)
error_rows = pd.DataFrame({"test_index": sampled_errors,
                            "true_label": y_test[sampled_errors],
                            "pred_label": cal_p.argmax(1)[sampled_errors],
                            "confidence": cal_p.max(1)[sampled_errors],
                            "text_excerpt": [test_df.iloc[i].text[:220] for i in sampled_errors]})
error_rows.to_csv(OUT / "sampled_errors.csv", index=False)
display(error_rows)

## 14. Frozen-model evaluation on an independent sentence-level dataset (SST-2)

**Secondary transfer endpoint:** Without **any** SST-2 training or parameter/hyperparameter selection, evaluate the saved IMDb BERT checkpoints and full-text TF–IDF comparator on the **872 labeled examples in the SST-2 validation split**. The public SST-2 test labels are hidden and therefore are not used. We use SST-2's movie-review **sentences**, which are shorter and differently annotated than IMDb's review **documents**. This probes sensitivity to granularity and dataset conventions, *not* general transfer to unrelated domains. Validation-only temperatures fitted on IMDb stay frozen when transferred. SST-2 provenance: https://huggingface.co/datasets/stanfordnlp/sst2 and Socher et al. (2013), https://aclanthology.org/D13-1170/.

For full reproducibility, this secondary benchmark must remain untouched by further hyperparameter/model-method decisions. Its dataset card currently lists an unknown license; check redistribution rights before releasing any derived text. The same metric definitions apply. The displayed cross-dataset performance is **not** claimed as state of the art.

In [ ]:
sst = load_dataset("stanfordnlp/sst2", split="validation")
sst_df = pd.DataFrame({"text": [normalize_imdb(s) for s in sst["sentence"]],
                       "label": [int(l) for l in sst["label"]]})
assert set(sst_df.label.unique()) == {0, 1}
sst_ds = ReviewDataset(sst_df, tokenizer, CFG.max_length)
sst_y = sst_df.label.to_numpy(dtype=int)
external_rows = []
for seed in CFG.train_seeds:
    frozen_model = AutoModelForSequenceClassification.from_pretrained(
        OUT / "checkpoints" / f"bert_seed_{seed}").to(DEVICE)
    sst_logits, y_check = forward_logits(frozen_model, sst_ds)
    assert np.array_equal(y_check, sst_y)
    frozen_prob = softmax(sst_logits / seed_outputs[seed]["temperature"], axis=1)
    external_rows.append({"system": "BERT_IMDb_frozen", "seed": seed,
                          **classification_metrics(sst_y, frozen_prob)})
    del frozen_model
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

# Lexical baseline uses the exact same IMDb-fitted feature vocabulary and weights.
lex_obj = lexical["full"]
sst_lex_prob = lex_obj["clf"].predict_proba(
    lex_obj["vectorizer"].transform(sst_df.text.tolist()))
external_rows.append({"system": "TFIDF_IMDb_frozen", "seed": "fixed",
                      **classification_metrics(sst_y, sst_lex_prob)})
external_df = pd.DataFrame(external_rows)
external_df.to_csv(OUT / "sst2_sentence_transfer.csv", index=False)
print("SST-2 labeled validation rows:", len(sst_df))
display(external_df.round(4))


## 15. Research result manifest, checkpoint usage, and reproducible inference

This section exports a complete results summary, runnable model artifacts, chosen development protocol, and a reusable inference-and-explanation helper. The function below **does not** claim calibrated explanations: IG always targets the model's original logit, while probabilities shown to the user can be temperature-scaled. Logit attribution ranking is unchanged by dividing all logits by a positive scalar temperature, but probability changes under masking are computed on the uncalibrated classifier in the faithfulness study. Do not mix calibrated and raw perturbation probabilities without relabeling the metric.

To make a later paper reproducible, archive this notebook, the exact package versions, selected checkpoints, data hashes and all generated CSVs; report hardware, seed-specific numbers, and any deviations from the stated protocol. Releasing fine-tuned model weights should comply with model/dataset licenses.

In [ ]:
def predict_and_explain(review: str, show=True):
    """Inference for the predeclared explanation-seed checkpoint."""
    review = normalize_imdb(review)
    exp = explain_text(review)
    x, _, _ = encode_for_explanation(review)
    with torch.inference_mode():
        logits = model(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).logits
        probability = torch.softmax(logits / seed_outputs[EXPLAIN_SEED]["temperature"], -1)[0]
    label = ["negative", "positive"][exp["prediction"]]
    summary = {"predicted_sentiment": label,
               "calibrated_positive_probability": float(probability[1].item()),
               "calibrated_predicted_class_probability": float(probability[exp["prediction"]].item()),
               "top_supporting_groups": [g["word"] for g in sorted(
                   exp["ig_groups"], key=lambda g: -g["score"])[:10]],
               "ig_completeness_absolute_residual": exp["ig_abs_delta"]}
    if show:
        print(json.dumps(summary, indent=2))
        display(render_word_highlights(review, exp["ig_groups"]))
    return summary

_ = predict_and_explain("The acting was exquisite, but the story was painfully dull and predictable.")

summary = {
    "title": "Explainable Sentiment Analysis using BERT",
    "config": asdict(CFG), "data_manifest": provenance,
    "resolved_environment": VERSIONS,
    "hypotheses": ["BERT vs full and truncated TF-IDF controls",
                   "IG faithfulness vs GradientXActivation and matched random",
                   "validation-only temperature scaling", "head-weight diagnostic"],
    "classification": results_df.to_dict(orient="records"),
    "sst2_sentence_level_transfer": external_df.to_dict(orient="records"),
    "bootstrap": ci_df.to_dict(orient="records"),
    "faithfulness_intervals": faith_comparison.to_dict(orient="records"),
    "ig_completeness": delta_df.ig_abs_delta.describe().to_dict(),
    "limitations": ["No human-annotated rationales on IMDb",
                    "Masking interventions create distribution shift",
                    "Attribution evaluation only for one predeclared trained seed",
                    "Cascading randomization and retraining with shuffled labels not included",
                    "Head truncation, benchmark saturation, possible pretraining overlap",
                    "Single development split and only three fine-tuning seeds"]
}
(OUT / "research_manifest.json").write_text(json.dumps(summary, indent=2, default=str))
print("Exported artifact files:")
for p in sorted(OUT.rglob("*")):
    if p.is_file() and len(p.relative_to(OUT).parts) <= 2:
        print(" -", p)

## 16. Critical discussion and requirements for an ICML-level submission

### What the results would and would not establish

1. **Predictive contribution:** A positive paired BERT-vs-TFIDF accuracy interval indicates a test-set improvement conditional on these fitted models and the IMDb sampling assumptions. It **does not** establish robustness to different domains, languages, time periods, or socially meaningful subgroups.
2. **Faithfulness:** If IG yields larger comprehensiveness *and* smaller sufficiency than both controls at matched budgets, that is evidence of useful rankings **under this [MASK] intervention**. It does not prove that those words are causes of the human label or that the ranking remains optimal with coherent natural-language removal.
3. **Completeness:** A small IG convergence residual supports numerical accuracy of the line integral; it does **not** validate semantic faithfulness. Increase `ig_steps` or change the baseline if the residual is problematic; disclose both.
4. **Calibration:** Temperature scaling is a lightweight probability adjustment, not a correction for dataset shift or misclassification.
5. **Sentence-level transfer:** SST-2 validation assesses a format/annotation shift within movie sentiment, not a broad out-of-domain or multilingual guarantee. There is no SST-2 fine-tuning or model selection.
6. **Statistical scope:** Review-level bootstrap intervals ignore training-seed uncertainty. Three training seeds are a useful start, not a general conclusion about algorithm stability.

### Concrete extension plan for a genuinely new research paper

A publishable methodological claim could concern **distribution-preserving, semantically constrained perturbations for token-level BERT explanations**, tested against standard `[MASK]` interventions. Pre-register: a new perturbation operator (e.g., grammatical counterfactual infilling plus label-preservation checks), a new score/optimization rule or theoretical property, and falsifiable predictions relative to IG and established rationale methods. Measure changes on **external datasets with human rationales** (e.g. ERASER movie rationales), multi-seed training, multi-model architectures, sequence lengths, and unshifted counterfactual evaluations. Report stability across tokenizers and baselines, human rationale agreement, an intervention-quality audit, confidence intervals at the appropriate hierarchical level, ablations, and false/negative findings. Make comparisons against well-tuned current methods under equal budgets and code/compute disclosure.

**Limitations to keep in any manuscript:** Movie sentiment is a narrow task; BERT-base is an established baseline; a single explanation score can mix lexical evidence, tokenization artifacts, and model-specific quirks; explanation visualization is vulnerable to confirmation bias; arbitrary mask perturbations may be off-distribution. The presented implementation makes **no** novel-method or SOTA claim.

### Citation-ready primary references

Devlin et al. (2019), NAACL, https://aclanthology.org/N19-1423/; Maas et al. (2011), ACL, https://aclanthology.org/P11-1015/; Sundararajan et al. (2017), ICML, https://proceedings.mlr.press/v70/sundararajan17a.html; DeYoung et al. (2020), ACL, https://aclanthology.org/2020.acl-main.408/; Adebayo et al. (2018), NeurIPS, https://arxiv.org/abs/1810.03292.